<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/LangStudio/RSI_Max_Backtesting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import yfinance as yf
import numpy as np
import seaborn as sns
import warnings
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

In [9]:
warnings.filterwarnings("ignore")

# --- 1. Configuration & Symbol Loading ---
# Bearish thresholds (Overbought levels)
RSI_LEVELS = [60, 65, 70, 75, 80]
RSI_LENGTHS = [3, 5, 7, 10, 12, 14, 16, 18, 22, 24, 26]

try:
    # Using your specific filename mentioned in previous turns
    csv_name = "OptionVolume50_20260201.csv"
    df_csv = pd.read_csv(csv_name)
    symbol_col = [c for c in df_csv.columns if 'symbol' in c.lower() or 'ticker' in c.lower()][0]
    SYMBOLS = df_csv[symbol_col].str.strip().unique().tolist()
    print(f"Loaded {len(SYMBOLS)} symbols from {csv_name}")
except Exception as e:
    print(f"Could not load CSV: {e}. Falling back to default list.")
    SYMBOLS = ["TSLA", "SPY", "QQQ", "NVDA", "AAPL", "MSFT"]

# --- 2. Enhanced RSI Function ---
def calculate_rsi_yahoo(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

# --- 3. Data Fetching ---
print("Fetching data and calculating vectorized returns...")
data_cache = {}
for s in SYMBOLS:
    try:
        # auto_adjust=True handles splits/dividends; multi_level=False prevents MultiIndex
        df = yf.download(s, period="5y", interval="1d", progress=False, auto_adjust=True)
        if df.empty: continue

        # Ensure column names are clean
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        # Vectorized Future Returns (The "Trade" result)
        df['Ret_3D'] = df['Close'].pct_change(3).shift(-3)
        df['Ret_5D'] = df['Close'].pct_change(5).shift(-5)
        df['Ret_10D'] = df['Close'].pct_change(10).shift(-10)

        data_cache[s] = df
    except:
        continue

# --- 4. Parameter Sweep ---
all_results = []

for rsi_len in RSI_LENGTHS:
    for rsi_thresh in RSI_LEVELS:
        param_trades = []

        for symbol, df_orig in data_cache.items():
            df = df_orig.copy()
            df['RSI'] = calculate_rsi_yahoo(df['Close'], period=rsi_len)

            # --- UPDATED TRIGGER LOGIC: THE CROSSOVER ---
            # 1. Current RSI is above threshold
            # 2. Previous RSI was below or equal to threshold
            condition = (df['RSI'] > rsi_thresh) & (df['RSI'].shift(1) <= rsi_thresh)

            trades = df[condition].dropna(subset=['Ret_10D']).copy()

            if not trades.empty:
                trades['RSI_Len'] = rsi_len
                trades['RSI_Thresh'] = rsi_thresh
                param_trades.append(trades)

        if param_trades:
            combined = pd.concat(param_trades)
            # BEARISH WIN RATE: A win is when future return is NEGATIVE (< 0)
            all_results.append({
                "RSI_Len": rsi_len,
                "RSI_Thresh": rsi_thresh,
                "Avg_3D_Ret": combined['Ret_3D'].mean(),
                "Avg_5D_Ret": combined['Ret_5D'].mean(),
                "Avg_10D_Ret": combined['Ret_10D'].mean(),
                "Win_Rate_3D": (combined['Ret_3D'] < 0).mean(), # Win = Price Down
                "Win_Rate_5D": (combined['Ret_5D'] < 0).mean(),
                "Win_Rate_10D": (combined['Ret_10D'] < 0).mean(),
                "Trade_Count": len(combined)
            })

# --- 5. Summary Display ---
if all_results:
    summary_df = pd.DataFrame(all_results)
    # Sorting by Win_Rate_5D to find the best bearish signals
    summary_df = summary_df.sort_values(by="Win_Rate_5D", ascending=False)

    print("\n--- BEARISH RSI OPTIMIZATION SUMMARY ---")
    display(summary_df)
    summary_df.to_csv("bearish_rsi_results.csv", index=False)
else:
    print("No bearish signals found.")

Loaded 50 symbols from OptionVolume50_20260201.csv
Fetching data and calculating vectorized returns...

--- BEARISH RSI OPTIMIZATION SUMMARY ---


,RSI_Len,RSI_Thresh,Avg_3D_Ret,Avg_5D_Ret,Avg_10D_Ret,Win_Rate_3D,Win_Rate_5D,Win_Rate_10D,Trade_Count
54,26,80,-0.013183,-0.003060,0.032641,0.625000,0.625000,0.525000,40
49,24,80,-0.013470,0.006785,0.034899,0.592593,0.518519,0.462963,54
44,22,80,-0.005557,0.008478,0.028706,0.589041,0.506849,0.397260,73
34,16,80,-0.006946,0.002314,0.025521,0.528497,0.502591,0.435233,193
53,26,75,0.000094,0.006147,0.039463,0.486339,0.497268,0.453552,183
39,18,80,-0.007971,0.006091,0.027700,0.507812,0.476562,0.468750,128
50,26,60,0.005539,0.007523,0.019018,0.471235,0.469942,0.443439,1547
29,14,80,0.016954,0.019960,0.029484,0.461818,0.465455,0.396364,275
10,7,60,0.006776,0.008112,0.014692,0.464386,0.463620,0.451111,3917
5,5,60,0.006233,0.008087,0.016309,0.462260,0.462464,0.451040,4902
